In [ ]:
import json
from pathlib import Path
from docling.backend.pypdfium2_backend import PyPdfiumDocumentBackend
from docling.datamodel.base_models import InputFormat
from docling.document_converter import (
    DocumentConverter,
    PdfFormatOption,
    WordFormatOption,
    ImageFormatOption
)
from docling.datamodel.pipeline_options import PdfPipelineOptions, EasyOcrOptions

from docling.pipeline.simple_pipeline import SimplePipeline
from docling.pipeline.standard_pdf_pipeline import StandardPdfPipeline



# 모듈 최상단에 패턴 컴파일
import re
NEWLINE_PATTERN = re.compile(r'\r\n\d+')

def normalize_newlines(text: str) -> str:
    """개행문자 정규화 (동기 함수)"""
    return NEWLINE_PATTERN.sub('\n', text)

import os
from time import sleep, time
import pickle
import pdfplumber
from tqdm.auto import tqdm
from langchain_core.documents import Document


# 공유 가능한 옵션 정의
DEFAULT_PIPELINE_OPTIONS = PdfPipelineOptions(
    do_ocr=True,
    do_table_structure=True,
    ocr_options=EasyOcrOptions(lang=["en", "ko"])
    )


def parsing_pdf_by_page_with_docling(path:str, lv1_cat:str, lv2_cat:str):
    path = path.replace("\\", "/")
    filename = path.split("/")[-1]

    first_sentence = f"This page explains {filename.replace(".pdf", "")} that belongs to {lv1_cat} and  {lv2_cat} categories.\n"


    pipeline_options = DEFAULT_PIPELINE_OPTIONS
    converter = DocumentConverter(
        allowed_formats=[
            InputFormat.PDF
        ],
        format_options={
            InputFormat.PDF: PdfFormatOption(
                pipeline_options=pipeline_options,
                backend=PyPdfiumDocumentBackend
            ),}
    )
    loaded_docs = converter.convert(path)
    with pdfplumber.open(path) as pdf:
        page_num = 0
        docs = []
        for _ in tqdm(pdf.pages):
            docling_text = loaded_docs.document.export_to_markdown(page_no=int(page_num)+1)
            docling_text = docling_text.replace("<!-- image -->", "")
            docling_text = normalize_newlines(docling_text)
            docling_text = first_sentence + docling_text
            lang_doc = Document(page_content=docling_text, metadata={'filename': filename, 'lv1_cat': lv1_cat, 'lv2_cat': lv2_cat, 'page':str(page_num)})
            docs.append(lang_doc)
            page_num+=1
            sleep(0.1)
    
    parsed_foldername = f"{lv1_cat}_{lv2_cat}"
    if not os.path.exists(f"./docs/{parsed_foldername}"):
        os.makedirs(f"./docs/{parsed_foldername}")
        
    parsed_filename = filename.replace(".pdf", "")
    with open(f"./docs/{parsed_foldername}/{parsed_filename}.pkl", 'ab') as file:
        pickle.dump(docs, file)

    # if os.path.exists(path):
    #     os.remove(path)

    return docs

In [14]:
start_time = time()
path = f"./file/kubota_sample.pdf"
lv1_cat, lv2_cat = "CE", "KUBOTA"
result = parsing_pdf_by_page_with_docling(path=path, lv1_cat=lv1_cat, lv2_cat=lv2_cat)
end_time = time() - start_time
print(f"Document converted and tables exported in {end_time:.2f} seconds.")

2026-01-18 12:35:52,321 - INFO - detected formats: [<InputFormat.PDF: 'pdf'>]
2026-01-18 12:35:52,371 - INFO - Going to convert document batch...
2026-01-18 12:35:52,372 - INFO - Initializing pipeline for StandardPdfPipeline with options hash 261a9e0a90b09b315d0544b49c103d05
2026-01-18 12:35:52,374 - INFO - Accelerator device: 'cpu'
2026-01-18 12:35:54,578 - INFO - Accelerator device: 'cpu'
2026-01-18 12:35:56,078 - INFO - Accelerator device: 'cpu'
2026-01-18 12:35:56,516 - INFO - Processing document kubota_sample.pdf
d:\auto_vectordb\.venv\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
d:\auto_vectordb\.venv\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
d:\auto_vectordb\.venv\Lib\sit

Document converted and tables exported in 247.17 seconds.


In [15]:
with open('./docs/CE_KUBOTA/kubota_sample.pkl', 'rb') as file:
    # Load the pickled object from the file
    loaded_object = pickle.load(file)

In [20]:
from IPython.display import Markdown
Markdown(loaded_object[5].page_content)

This page explains kubota_sample that belongs to CE and  KUBOTA categories.
## SPECIFICATIONS

| Model                                                    | Model                                                    | Model                                                    | Model                                                    | Model                                                    | KX015-4                               |
|----------------------------------------------------------|----------------------------------------------------------|----------------------------------------------------------|----------------------------------------------------------|----------------------------------------------------------|---------------------------------------|
| Machine weight *1  (cabin / canopy)                      | Machine weight *1  (cabin / canopy)                      | Machine weight *1  (cabin / canopy)                      | Machine weight *1  (cabin / canopy)                      | kg                                                       | 1470 / 1420                           |
| Operating weight *2  (cabin / canopy)                    | Operating weight *2  (cabin / canopy)                    | Operating weight *2  (cabin / canopy)                    | Operating weight *2  (cabin / canopy)                    | kg                                                       | 1545 / 1495                           |
|                                                          | Model                                                    | Model                                                    | Model                                                    | Model                                                    | D782-E3-BH                            |
|                                                          | Type                                                     | Type                                                     | Type                                                     | Type                                                     | Water- r- cooled,diesel engine,E-TVCS |
|                                                          | Output ISO14396                                          | Output ISO14396                                          | Output ISO14396                                          | PS (kW)/rpm                                              | 13.3 (9.8) / 2300                     |
| Engine                                                   | Output ISO9249 NET                                       | Output ISO9249 NET                                       | Output ISO9249 NET                                       | PS (kW)/rpm                                              | 13.1 (9.6) / 2300                     |
|                                                          | Number of cylinders                                      | Number of cylinders                                      | Number of cylinders                                      | Number of cylinders                                      | 3                                     |
|                                                          | Bore × Stroke                                            | Bore × Stroke                                            | Bore × Stroke                                            | mm                                                       | 67 × 73.6                             |
|                                                          | Displacement                                             | Displacement                                             | Displacement                                             | cc                                                       | 778                                   |
|                                                          | Overall width                                            | Overall width                                            | Overall width                                            | mm                                                       | 990                                   |
|                                                          | Overall height (cabin / canopy)                          | Overall height (cabin / canopy)                          | Overall height (cabin / canopy)                          | mm                                                       | 2350 / 2330                           |
|                                                          | Overall length                                           | Overall length                                           | Overall length                                           | mm                                                       | 3710                                  |
|                                                          | Ground clearance                                         | Ground clearance                                         | Ground clearance                                         | mm                                                       | 160                                   |
| Dimensions                                               | Dozer size (width × height)                              | Dozer size (width × height)                              | Dozer size (width × height)                              | mm                                                       | 990 × 230                             |
|                                                          | Rubber shoe width                                        | Rubber shoe width                                        | Rubber shoe width                                        | mm                                                       | 230                                   |
|                                                          | Minimum front swivel radius                              | Minimum front swivel radius                              | Minimum front swivel radius                              | mm                                                       | 1490                                  |
|                                                          | Boom swing angle (left /right) deg                       | Boom swing angle (left /right) deg                       | Boom swing angle (left /right) deg                       | Boom swing angle (left /right) deg                       | 75 / 60                               |
| Hydraulic                                                | P1, P2                                                   |                                                          |                                                          |                                                          | Variable displacement pump            |
| Hydraulic                                                |                                                          | Flow rate /min                                           | Flow rate /min                                           | Flow rate /min                                           | 16.6 × 2                              |
| Hydraulic                                                |                                                          | Hydraulic pressure MPa (kgf/cm 2 )                       | Hydraulic pressure MPa (kgf/cm 2 )                       | Hydraulic pressure MPa (kgf/cm 2 )                       | 20.6 (210)                            |
| Hydraulic                                                | P3                                                       |                                                          |                                                          |                                                          | Gear pump                             |
| Hydraulic                                                |                                                          | Flow rate /min                                           | Flow rate /min                                           | Flow rate /min                                           | 10.4                                  |
| System                                                   |                                                          | Hydraulic pressure MPa (kgf/cm 2 )                       | Hydraulic pressure MPa (kgf/cm 2 )                       | Hydraulic pressure MPa (kgf/cm 2 )                       | 20.1 (205)                            |
| Hydraulic                                                | Auxiliary                                                | /min Max. flow rate                                      | /min Max. flow rate                                      | /min Max. flow rate                                      | 27.0                                  |
| Hydraulic                                                | (AUX)                                                    | MPa (kgf/cm 2 ) Max. hydr. pressure                      | MPa (kgf/cm 2 ) Max. hydr. pressure                      | MPa (kgf/cm 2 ) Max. hydr. pressure                      | 20.6 (210)                            |
| Hydraulic                                                | Max. digging                                             |                                                          | arm                                                      | kN (kgf)                                                 | 7.3 (740)                             |
| Hydraulic                                                | force                                                    |                                                          | bucket                                                   | kN (kgf)                                                 | 12.7 (1300)                           |
| Hydraulic                                                | Hydraulic reservoir (full)                               | Hydraulic reservoir (full)                               | Hydraulic reservoir (full)                               | Hydraulic reservoir (full)                               | 28                                    |
| Max. travelling speed  km/h                              | Max. travelling speed  km/h                              | Max. travelling speed  km/h                              | Max. travelling speed  km/h                              | Max. travelling speed  km/h                              | 2.1                                   |
| Ground contact pressure (cabin / canopy) kPa (kgf/cm 2 ) | Ground contact pressure (cabin / canopy) kPa (kgf/cm 2 ) | Ground contact pressure (cabin / canopy) kPa (kgf/cm 2 ) | Ground contact pressure (cabin / canopy) kPa (kgf/cm 2 ) | Ground contact pressure (cabin / canopy) kPa (kgf/cm 2 ) | 25.5 (0.26) / 24.5 (0.25)             |
| Swivelling speed rpm                                     | Swivelling speed rpm                                     | Swivelling speed rpm                                     | Swivelling speed rpm                                     | Swivelling speed rpm                                     | 9.1                                   |
| Fuel tank capacity                                       | Fuel tank capacity                                       | Fuel tank capacity                                       | Fuel tank capacity                                       | Fuel tank capacity                                       | 21                                    |
| LpA dB (A)                                               | LpA dB (A)                                               | LpA dB (A)                                               | LpA dB (A)                                               | LpA dB (A)                                               | 78                                    |
| Noise level                                              | LwA (2000/14/EC)                                         |                                                          |                                                          | dB (A)                                                   | 93                                    |
|                                                          | Hand arm system (ISO5349-2:2001)                         |                                                          | Digging                                                  | m/s2 RMS                                                 | <2.5                                  |
|                                                          |                                                          |                                                          | Levelling                                                | m/s2 RMS                                                 | <2.5                                  |
|                                                          |                                                          |                                                          | Driving                                                  | m/s2 RMS                                                 | <2.5                                  |
| Vibration* KX0154  n*3 Whole  Vibration KX015-4 canopy                                                          |                                                          |                                                          | Idling                                                   | m/s2 RMS                                                 | <2.5                                  |
|                                                          | Whole body anopy (ISO2631-1:1997)                                                          |                                                          | Digging                                                  | m/s2 RMS m/s2 RMS oint radiu ing m/s2 RMS Lifting point radius (2m)                                                          | <0.5 <0.5                             |
|                                                          | Lift Point Height                                        | Levelling Lifti                                                          | Driving Ove ving m Over-front                                                          | m/s2 RMS t                                                          | <0.5                                  |
|                                                          |                                                          | Idling Blade Down                                                          | Idling Down                                                          | m/s2 RMS Blade UP m/s RMS Blade UP                                                          | <0.5 de <0 Over-side Blade Down                                       |

*1 With 32.5 kg Kubota original bucket, full tanks, rubber shoe. 
 (035)
 (030)
m ubber shoe
.4 (0.35)

*2 Machine weight with 75 kg operator.
.0 (0.30
.5m al bucket, fu
operator.
.0 (0.30) 2.5 kg 
ne wei
.5m

2.7 (0.27)

–

*3 These values are mesured under specific conditions at maximum engine speed and can deviate, 
 (037)
 (025)
–
 (046)
m depending on the operating status.
 (0 itions at ma
.6 (0.37) ngine speed 
.5 (0.25) – d under spe
g status
.5 (0.46) values 
.0m

0.5m

3.4 (0.35)

3.3 (0.33)
Y

2.3 (0.23)

2.2 (0.22)

2.6 (0.27)

–

Blade UP

–

–

1.7 (0.17)

–

5.4 (0.55)

## 5.3 (0.54)
APAC 0m
TIN 3.
.3 (0.54)
m
LIFTING CAPACITY

Lifting point radius (max.)

Over-front

Over-side

–

–

1.1 (0.12)

–

| KX015-4 cabin Cabin, Rubber v KX015-4 cabin Cabin, Rubber version                   |                           |                           |                           |                             |                             | kN (ton)                    |
|-------------------|---------------------------|---------------------------|---------------------------|-----------------------------|-----------------------------|-----------------------------|
|                   | Lifting point radius (2m) | Lifting point radius (2m) | Lifting point radius (2m) | Lifting point radius (max.) | Lifting point radius (max.) | Lifting point radius (max.) |
| Lift Point Height | Over-front                | Over-front                | Over-side                 | Over-front                  | Over-front                  | Over-side Pit                             |
|                   | Blade Down                | Blade UP                  |                           | Blade Down                  | Blade UP                    | Over Lift Point                             |
| 1.5m              | 3.0 (0.30)                | 3.4 (0.35)                | 2.5 (0.26)                | –                           | –                           | –                           |
| 1.0m              | 4.5 (0.46)                | 3.5 (0.36)                | 2.3 (0.24)                | –                           | –                           | –                           |
| 0.5m              | 5.4 (0.55)                | 3.3 (0.34)                | 2.1 (0.22)                | 2.6 (0.27)                  | 1.6 (0.17)                  | 1.1 (0.11) oint Heigh 1.1 (0.11) Lift Point Height                             |
| 0m                | 5.3 (0.54)                | 3.2 (0.32)                | 2.0 (0.21)                | –                           | –                           | –                           |

*The lifting capacities are based on ISO 10567 and do not exceed 75% of the static tilt load of the machine or 87%

of the hydraulic lifting capacities of the machine.

**The excavator bucket, hook, sling and other lifting accessories are not included on this table.



* Working ranges are with Kubota original bucket, without quick coupler.

* Specifications are subject to change without notice for purpose of improvement.

- ★ All images shown are for brochure purposes only. When operating the excavator, wear clothing and equipment in accordance to local legal and safety regulations.

## KUBOTA (U.K.) LTD

Dormer Road,Thame, Oxfordshire, OX9 3UN, U.K.

Phone : 01844-268140

F a x : 01844-216685



3360

2250

## WORKING RANGE

3730

3790



1490



450

510

450

510

*With cabin, rubber shoe and standard arm kN (ton)

2290

1810

990

1490

1090

1450

1090

3710

990

K


240

230

1070


1070
KX15-4\_2010/12


990

990

240

230

2350

2350

KX15-4\_2010/12

KX15-4\_2010/12

In [ ]:
with open('./docs/SHIP_GENERATOR/FWG.pkl', 'rb') as file:
    # Load the pickled object from the file
    loaded_object = pickle.load(file)

In [12]:
from IPython.display import Markdown
Markdown(loaded_object[1].page_content)

This page explains FWG that belongs to SHIP and  GENERATOR categories.


| PACKAGE LIST   | PACKAGE LIST   | PACKAGE LIST                           |      |                    |
|----------------|----------------|----------------------------------------|------|--------------------|
| POR NO.        | POR NO.        | HULL NO. : 8250/8251                   |      |                    |
| SER.  NO.      | SEQ.  NO.      | DESCRIPTION                            | Q’TY | REMARK             |
| M536           | AA             | F.W. GENERATOR EVAPORATING TYPE        | 1    | Separately  packed |
| M536           | BB             | SPARE PARTS & TOOLS FOR F.W. GENERATOR | 1    |                    |

1. EACH UNIT WITHIN A PACKAGE OR SHIPPING CONTAINER SHALL BE CLEARLY MARKED IN A MANNER AS MAY BE DESIGNATED BY THE BUYER BY STAMPING, TAGGING OR OTHER SUITABLE MEANS WITH IDENTIFICATION OF SUPPLY.

THE OUTSIDE OF EACH PACKAGE AND OR PROTECTIVE DEVICES SHALL BE CLEARLY MARKED, REFERING SHIPPING MARK.

2. ABOVE POR NO. (SER. NO. - SEQ. NO.) AND DESCRIPTION MUST BE MARKED ON EACH PACKAGE AND PACKING LIST.

/HHIPN301584/2024-01-25 PM 06:28:29(2/7)

자재구매부/이택용